In [1]:
import scipy.io
import torch
import numpy as np
import pandas as pd
from chronos import Chronos2Pipeline
import pickle as pl
import glob
import traceback
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


/Users/adityagoyal/miniconda3/envs/baseline_env/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/Users/adityagoyal/miniconda3/envs/baseline_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ekpping a buffer for about 1 hour for the gap between vt and birth and vt and age at death

In [ ]:
import scipy.io
import torch
import numpy as np
import pandas as pd
from chronos import Chronos2Pipeline
import pickle as pl
import glob
import traceback
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# Load pipeline
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", 
                                            device_map="cpu", 
                                            dtype=torch.bfloat16)

# Constants
CONTEXT_SIZE_HOURS = 80
CONTEXT_SIZE_SECONDS = CONTEXT_SIZE_HOURS * 3600  # 288000 seconds
WINDOW_SIZE_SECONDS = 300  # 5 minutes
NUM_WINDOWS = CONTEXT_SIZE_SECONDS // WINDOW_SIZE_SECONDS  # 960 windows
THRESHOLDS_DAYS = [3, 5, 7]
THRESHOLDS_SECONDS = [d * 86400 for d in THRESHOLDS_DAYS]

# ============================================================================
# CONFIGURATION: Number of datapoints to generate
# ============================================================================
NUM_LABEL_0_NICU = 10        # Label 0 windows for NICU babies (before threshold)
NUM_LABEL_1_NICU = 10        # Label 1 windows for NICU babies (within threshold)
NUM_LABEL_0_NON_NICU = 10    # Label 0 windows for non-NICU babies (survived)

def create_windowed_features(mat_file_path, start_time, context_size, window_size):
    """Create windowed features with NaN filtering"""
    mat_data = scipy.io.loadmat(mat_file_path)
    
    # Get HR and SPO2 columns (assuming positions 0 and 2)
    hr_cols = mat_data['vdata'][:,0]
    spo2_cols = mat_data['vdata'][:,2]
    timestamps = mat_data['vt'].flatten()
    
    # Filter by time range AND valid data
    end_time = start_time + context_size
    time_mask = (timestamps >= start_time) & (timestamps < end_time)
    valid_mask = ~np.isnan(hr_cols) & ~np.isnan(spo2_cols)
    final_mask = time_mask & valid_mask
    
    timestamps_valid = timestamps[final_mask]
    hr_valid = hr_cols[final_mask]
    spo2_valid = spo2_cols[final_mask]
    
    if len(timestamps_valid) == 0:
        return None
    
    # Assign each timestamp to a window
    window_indices = np.floor((timestamps_valid - start_time) / window_size).astype(int)
    num_windows = int(np.ceil(context_size / window_size))
    
    # Aggregate using mean
    hr_windows = np.full(num_windows, np.nan)
    spo2_windows = np.full(num_windows, np.nan)
    
    for idx in range(num_windows):
        mask = window_indices == idx
        if np.any(mask):
            hr_windows[idx] = np.mean(hr_valid[mask])
            spo2_windows[idx] = np.mean(spo2_valid[mask])
    
    return hr_windows, spo2_windows

def get_embedding(hr_windows, spo2_windows, pipeline):
    """Get Chronos embedding from windowed data - Fixed for BFloat16"""
    # Convert NaN-filled arrays to float32 (handle NaN with mean imputation)
    hr_clean = np.where(np.isnan(hr_windows), np.nanmean(hr_windows) if not np.all(np.isnan(hr_windows)) else 0, hr_windows).astype(np.float32)
    spo2_clean = np.where(np.isnan(spo2_windows), np.nanmean(spo2_windows) if not np.all(np.isnan(spo2_windows)) else 0, spo2_windows).astype(np.float32)
    
    hr_tensor = torch.tensor(hr_clean, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    spo2_tensor = torch.tensor(spo2_clean, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    context = torch.cat([hr_tensor, spo2_tensor], dim=0)  # (2, 1, num_windows)
    
    with torch.no_grad():
        embeddings, _ = pipeline.embed(context)
    
    # Convert embeddings from bfloat16 to float32 before numpy conversion
    hr_emb = embeddings[0][0][-1].float()
    spo2_emb = embeddings[1][0][-1].float()
    return torch.cat([hr_emb, spo2_emb], dim=0).cpu().numpy()

def generate_non_nicu_embeddings(mat_file_path, num_windows=NUM_LABEL_0_NON_NICU):
    """
    Generate label 0 embeddings for non-NICU babies (survived)
    
    Returns up to `num_windows` non-overlapping context windows, each label 0
    """
    mat_data = scipy.io.loadmat(mat_file_path)
    birth_time = mat_data['pdata'][0][1]
    
    if np.isnan(birth_time):
        return []
    
    timestamps = mat_data['vt'].flatten()
    last_ts = timestamps[-1].item()
    
    datapoints = []
    
    # Generate non-overlapping windows from birth to end of recording
    available = last_ts - birth_time
    if available >= CONTEXT_SIZE_SECONDS:
        max_windows = min(num_windows, int(available / CONTEXT_SIZE_SECONDS))
        step = available / max_windows
        
        for i in range(max_windows):
            start = birth_time + i * step
            result = create_windowed_features(mat_file_path, start, 
                                             CONTEXT_SIZE_SECONDS, WINDOW_SIZE_SECONDS)
            if result is not None:
                embedding = get_embedding(result[0], result[1], pipeline)
                datapoints.append((embedding, 0))
    
    return datapoints

def generate_nicu_embeddings(mat_file_path, threshold_seconds, 
                             num_label_0=NUM_LABEL_0_NICU, 
                             num_label_1=NUM_LABEL_1_NICU):
    """
    Generate embeddings for NICU babies (died) - threshold-dependent
    
    Returns:
        - Up to `num_label_0` OVERLAPPING windows BEFORE threshold (label 0)
        - Up to `num_label_1` OVERLAPPING windows WITHIN threshold period (label 1)
    """
    mat_data = scipy.io.loadmat(mat_file_path)
    age_at_death = mat_data['pdata'][0][-1]
    birth_time = mat_data['pdata'][0][1]
    
    if np.isnan(age_at_death) or np.isnan(birth_time):
        return []
    
    death_ts = age_at_death * 86400
    threshold_start = death_ts - threshold_seconds
    
    datapoints = []
    
    # Label 0: OVERLAPPING windows BEFORE threshold
    available = threshold_start - birth_time
    if available >= CONTEXT_SIZE_SECONDS and num_label_0 > 0:
        # Create overlapping windows
        step = (available - CONTEXT_SIZE_SECONDS) / (num_label_0 - 1) if num_label_0 > 1 else 0
        
        for i in range(num_label_0):
            start = birth_time + i * step
            result = create_windowed_features(mat_file_path, start, 
                                             CONTEXT_SIZE_SECONDS, WINDOW_SIZE_SECONDS)
            if result is not None:
                embedding = get_embedding(result[0], result[1], pipeline)
                datapoints.append((embedding, 0))
    
    # Label 1: OVERLAPPING windows WITHIN threshold period
    if num_label_1 > 0:
        # Create overlapping windows ending at death
        # Step backward from death to create overlapping windows
        step = (threshold_seconds - CONTEXT_SIZE_SECONDS) / (num_label_1 - 1) if num_label_1 > 1 else 0
        step = max(step, 0)  # Ensure non-negative step (handles threshold < context)
        
        for i in range(num_label_1):
            # Start from (death - context_size) and step backward into threshold period
            start = death_ts - CONTEXT_SIZE_SECONDS - i * step
            
            # Ensure window is within valid range
            start = max(birth_time, start)  # Don't go before birth
            
            # Only add if window overlaps with threshold period
            if start + CONTEXT_SIZE_SECONDS > threshold_start:
                result = create_windowed_features(mat_file_path, start, 
                                                 CONTEXT_SIZE_SECONDS, WINDOW_SIZE_SECONDS)
                if result is not None:
                    embedding = get_embedding(result[0], result[1], pipeline)
                    datapoints.append((embedding, 1))
    
    return datapoints

def run_experiment(nicu_files, non_nicu_embeddings_label0, threshold_days, threshold_seconds):
    """
    Run experiment for one threshold
    
    Total datapoints per file:
    - NICU: up to (NUM_LABEL_0_NICU + NUM_LABEL_1_NICU) = 20 points
    - Non-NICU: up to NUM_LABEL_0_NON_NICU = 10 points (reused across thresholds)
    """
    print(f"\n{'='*60}")
    print(f"Threshold: <{threshold_days} days")
    print(f"{'='*60}")
    
    embeddings, labels = [], []
    error_count = 0
    
    # Add pre-computed non-NICU embeddings (label 0)
    for emb, lbl in non_nicu_embeddings_label0:
        embeddings.append(emb)
        labels.append(lbl)
    
    print(f"Added {len(non_nicu_embeddings_label0)} non-NICU label 0 embeddings")
    
    # Process NICU files (threshold-dependent)
    print(f"\nProcessing {len(nicu_files)} NICU files...")
    for i, file_path in enumerate(nicu_files):
        baby_id = file_path.split('/')[-1].split('_')[1]
        print(f"{i+1}/{len(nicu_files)}: Baby {baby_id}...", end=" ")
        
        try:
            datapoints = generate_nicu_embeddings(file_path, threshold_seconds)
            for emb, lbl in datapoints:
                embeddings.append(emb)
                labels.append(lbl)
            print(f"{len(datapoints)} points (label 0: {sum(1 for _, l in datapoints if l==0)}, label 1: {sum(1 for _, l in datapoints if l==1)})")
        except Exception as e:
            error_count += 1
            print(f"\nERROR: {str(e)}")
            traceback.print_exc()
    
    print(f"\n{'='*60}")
    print(f"Total: {len(embeddings)} datapoints")
    print(f"Label 0: {sum(l==0 for l in labels)} | Label 1: {sum(l==1 for l in labels)}")
    print(f"Errors: {error_count}")
    print(f"{'='*60}")
    
    with open(f'embeddings_{threshold_days}d.pkl', 'wb') as f:
        pl.dump({'embeddings': embeddings, 'labels': labels}, f)
    
    return embeddings, labels

# Dataset and Model
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = [torch.FloatTensor(e) for e in embeddings]
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

class Classifier(nn.Module):
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden // 2, 2)
        )
    
    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for embs, labels in loader:
        embs, labels = embs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(embs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, device):
    model.eval()
    preds, lbls, probs = [], [], []
    with torch.no_grad():
        for embs, labels in loader:
            embs, labels = embs.to(device), labels.to(device)
            outputs = model(embs)
            probs.extend(torch.softmax(outputs, 1)[:, 1].cpu().numpy())
            preds.extend(outputs.argmax(1).cpu().numpy())
            lbls.extend(labels.cpu().numpy())
    
    return {
        'accuracy': accuracy_score(lbls, preds),
        'precision': precision_score(lbls, preds, zero_division=0),
        'recall': recall_score(lbls, preds, zero_division=0),
        'f1': f1_score(lbls, preds, zero_division=0),
        'auc': roc_auc_score(lbls, probs) if len(set(lbls)) > 1 else 0
    }

def k_fold_cv(embeddings, labels, k=5, epochs=50, batch_size=32, lr=0.001, device='cpu'):
    """K-fold cross validation"""
    if len(embeddings) == 0:
        print("No embeddings to train on!")
        return None, None, None
    
    embeddings = np.array(embeddings)
    labels = np.array(labels)
    dim = embeddings[0].flatten().shape[0]
    
    print(f"\nK-Fold CV: {len(labels)} samples, {k} folds, embedding dim={dim}")
    print(f"Class distribution: {np.bincount(labels)}")
    
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(embeddings, labels)):
        print(f"Fold {fold+1}/{k}...", end=" ")
        
        train_data = EmbeddingDataset(embeddings[train_idx], labels[train_idx])
        val_data = EmbeddingDataset(embeddings[val_idx], labels[val_idx])
        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_data, batch_size=batch_size)
        
        model = Classifier(dim).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        for _ in range(epochs):
            train_epoch(model, train_loader, criterion, optimizer, device)
        
        metrics = validate(model, val_loader, device)
        fold_metrics.append(metrics)
        print(f"Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")
    
    # Average metrics
    avg = {k: np.mean([f[k] for f in fold_metrics]) for k in fold_metrics[0]}
    std = {k: np.std([f[k] for f in fold_metrics]) for k in fold_metrics[0]}
    
    print(f"\n{'='*60}")
    for k in avg:
        print(f"{k.capitalize():12s}: {avg[k]:.4f} ± {std[k]:.4f}")
    
    return avg, std, fold_metrics

# ============================================================================
# MAIN EXECUTION
# ============================================================================

print(f"CONFIGURATION:")
print(f"  NICU files - Label 0 windows: {NUM_LABEL_0_NICU}")
print(f"  NICU files - Label 1 windows: {NUM_LABEL_1_NICU}")
print(f"  Non-NICU files - Label 0 windows: {NUM_LABEL_0_NON_NICU}")
print(f"  Expected NICU datapoints per file: up to {NUM_LABEL_0_NICU + NUM_LABEL_1_NICU}")
print(f"  Expected non-NICU datapoints per file: up to {NUM_LABEL_0_NON_NICU}\n")

# Load valid baby IDs
print("Loading valid baby IDs from valid_summary_dic.pkl...")
with open('valid_summary_dic.pkl', 'rb') as f:
    valid_summary_dic = pl.load(f)

# Split into NICU and non-NICU
nicu_mask = valid_summary_dic['indicator_vector'] == 1
nicu_baby_ids = set(valid_summary_dic['baby_id'][nicu_mask])
non_nicu_baby_ids = set(valid_summary_dic['baby_id'][~nicu_mask])

print(f"NICU babies (died): {len(nicu_baby_ids)}")
print(f"Non-NICU babies (survived): {len(non_nicu_baby_ids)}")

# Filter mat files
all_mat_files = sorted(glob.glob('./download/Vitals/*.mat'))
nicu_files = [f for f in all_mat_files if f.split('/')[-1] in nicu_baby_ids]
non_nicu_files = [f for f in all_mat_files if f.split('/')[-1] in non_nicu_baby_ids]

print(f"\nNICU mat files: {len(nicu_files)}")
print(f"Non-NICU mat files: {len(non_nicu_files)}")

# OPTIMIZATION: Generate non-NICU embeddings ONCE
print(f"\n{'='*60}")
print("PRE-PROCESSING: Generating non-NICU embeddings (label 0)...")
print(f"{'='*60}")

non_nicu_embeddings_label0 = []
for i, file_path in enumerate(non_nicu_files):
    baby_id = file_path.split('/')[-1].split('_')[1]
    print(f"{i+1}/{len(non_nicu_files)}: Baby {baby_id}...", end=" ")
    
    try:
        datapoints = generate_non_nicu_embeddings(file_path)
        non_nicu_embeddings_label0.extend(datapoints)
        print(f"{len(datapoints)} points")
    except Exception as e:
        print(f"ERROR: {str(e)}")

print(f"\nTotal non-NICU label 0 embeddings: {len(non_nicu_embeddings_label0)}")

# Save for reuse
with open('non_nicu_embeddings_label0.pkl', 'wb') as f:
    pl.dump(non_nicu_embeddings_label0, f)

# Run experiments for each threshold
results = {}
for threshold_days, threshold_seconds in zip(THRESHOLDS_DAYS, THRESHOLDS_SECONDS):
    embeddings, labels = run_experiment(nicu_files, non_nicu_embeddings_label0, 
                                       threshold_days, threshold_seconds)
    
    if len(embeddings) > 0:
        avg, std, folds = k_fold_cv(embeddings, labels, k=5, epochs=50)
        results[threshold_days] = {'avg': avg, 'std': std, 'folds': folds}
        
        with open(f'results_{threshold_days}d.pkl', 'wb') as f:
            pl.dump(results[threshold_days], f)

# Final summary
print(f"\n{'='*60}\nFINAL SUMMARY\n{'='*60}")
for days in THRESHOLDS_DAYS:
    if days in results and results[days]['avg'] is not None:
        print(f"\nThreshold: <{days} days")
        for k, v in results[days]['avg'].items():
            print(f"  {k:12s}: {v:.4f} ± {results[days]['std'][k]:.4f}")

CONFIGURATION:
  NICU files - Label 0 windows: 10
  NICU files - Label 1 windows: 10
  Non-NICU files - Label 0 windows: 10
  Expected NICU datapoints per file: up to 20
  Expected non-NICU datapoints per file: up to 10

Loading valid baby IDs from valid_summary_dic.pkl...
NICU babies (died): 186
Non-NICU babies (survived): 4841

NICU mat files: 186
Non-NICU mat files: 4841

PRE-PROCESSING: Generating non-NICU embeddings (label 0)...
1/4841: Baby 1005... 0 points
2/4841: Baby 1007... 

In [ ]:
# precision and recall are calculated for the positive class (label 1) by default.


In [ ]:
import pickle as pl

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/results_3d.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
arr

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/results_5d.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
arr

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/results_7d.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
arr